# Statistical Significance Testing
Wilcoxon signed-rank tests on per-fold MAE across cities and horizons.

Comparisons:
1. TabPFN vs TabPFN_NoWeather (RQ2)
2. TabPFN vs XGBoost (RQ1)
3. NeuralProphet vs NeuralProphet_NoWeather (RQ2 contrast)

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon
from itertools import product
from pathlib import Path

RESULTS_VERSION = "v6"
DATASETS = ["seoul", "london", "washington"]

COMPARISONS = [
    ("TabPFN",        "TabPFN_NoWeather"),   # RQ2
    ("TabPFN",        "XGBoost"),             # RQ1
    ("NeuralProphet", "NeuralProphet_NoWeather"),  # RQ2 contrast
] 

# NOTE: I THINK MISUNDERSTANDING OF WHAT THE RESEARCH QUESTIONS ARE!

df_detailed = pd.read_csv(f'../results/detailed_results_master_{RESULTS_VERSION}.csv')
df_detailed = df_detailed[df_detailed['weather_scenario'] == 'clean_only'].copy()
print(f"Loaded: {df_detailed.shape}")
print(f"Models: {sorted(df_detailed['model'].unique())}")
print(f"Folds:  {sorted(df_detailed['fold'].unique())}")

Loaded: (58044, 17)
Models: ['ARIMA', 'NeuralProphet', 'NeuralProphet_NoWeather', 'NeuralProphet_NoWeather_Untuned', 'NeuralProphet_Untuned', 'Prophet', 'Prophet_Untuned', 'SARIMAX', 'Seasonal_Naive', 'TabPFN', 'TabPFN_NoWeather', 'TabPFN_NoWeather_v3', 'TabPFN_v3', 'XGBoost']
Folds:  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164

## 1. Per city × horizon tests
One Wilcoxon test per (city, horizon, comparison). Observations = per-fold MAE values.

In [2]:
def filter_results(df, version, dataset):
    df = df[df['version'] == version].copy()
    df = df[df['dataset'] == dataset].copy()
    return df

rows = []

for dataset, (model_a, model_b) in product(DATASETS, COMPARISONS):
    df_city = filter_results(df_detailed, RESULTS_VERSION, dataset)
    horizons = sorted(df_city['horizon'].unique())

    for h in horizons:
        df_h = df_city[df_city['horizon'] == h]

        a_vals = df_h[df_h['model'] == model_a].sort_values('fold')['MAE'].values
        b_vals = df_h[df_h['model'] == model_b].sort_values('fold')['MAE'].values

        if len(a_vals) == 0 or len(b_vals) == 0:
            continue
        if len(a_vals) != len(b_vals):
            print(f"WARNING: unequal folds for {model_a} vs {model_b} in {dataset} h={h}")
            continue

        diff = a_vals - b_vals
        if np.all(diff == 0):
            stat, p = np.nan, np.nan
        else:
            stat, p = wilcoxon(a_vals, b_vals, alternative='two-sided')

        rows.append({
            'comparison': f"{model_a} vs {model_b}",
            'model_a': model_a,
            'model_b': model_b,
            'dataset': dataset,
            'horizon': h,
            'n_folds': len(a_vals),
            'mean_MAE_a': a_vals.mean().round(2),
            'mean_MAE_b': b_vals.mean().round(2),
            'mean_diff': (a_vals - b_vals).mean().round(2),
            'statistic': stat,
            'p_value': p,
        })

results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

                              comparison       model_a                 model_b    dataset  horizon  n_folds  mean_MAE_a  mean_MAE_b  mean_diff  statistic      p_value
              TabPFN vs TabPFN_NoWeather        TabPFN        TabPFN_NoWeather      seoul        6      980      151.41      185.11     -33.70   163494.0 4.283703e-18
              TabPFN vs TabPFN_NoWeather        TabPFN        TabPFN_NoWeather      seoul       24      245      194.56      272.89     -78.33     5139.0 3.848156e-19
              TabPFN vs TabPFN_NoWeather        TabPFN        TabPFN_NoWeather      seoul       48      122      216.12      315.66     -99.54      632.0 1.582796e-15
              TabPFN vs TabPFN_NoWeather        TabPFN        TabPFN_NoWeather      seoul      168       35      222.25      313.90     -91.64       34.0 2.168235e-07
                       TabPFN vs XGBoost        TabPFN                 XGBoost      seoul        6      980      151.41      188.24     -36.84   157438.0 8.416901e-2

## 2. Multiple comparison correction (Holm-Bonferroni)
Applied separately per comparison pair.

In [3]:
from statsmodels.stats.multitest import multipletests

corrected_rows = []

for comp, grp in results_df.groupby('comparison'):
    grp = grp.copy().dropna(subset=['p_value'])
    reject, p_adj, _, _ = multipletests(grp['p_value'], method='holm')
    grp['p_adjusted'] = p_adj.round(4)
    grp['significant'] = reject
    corrected_rows.append(grp)

results_corrected = pd.concat(corrected_rows).reset_index(drop=True)

print(results_corrected[
    ['comparison','dataset','horizon','n_folds',
     'mean_MAE_a','mean_MAE_b','mean_diff',
     'p_value','p_adjusted','significant']
].to_string(index=False))

                              comparison    dataset  horizon  n_folds  mean_MAE_a  mean_MAE_b  mean_diff      p_value  p_adjusted  significant
NeuralProphet vs NeuralProphet_NoWeather      seoul        6      980      665.47      692.11     -26.64 2.761214e-21      0.0000         True
NeuralProphet vs NeuralProphet_NoWeather      seoul       24      245      375.43      386.67     -11.24 2.893900e-02      0.0629        False
NeuralProphet vs NeuralProphet_NoWeather      seoul       48      122      417.10      445.70     -28.60 2.277365e-07      0.0000         True
NeuralProphet vs NeuralProphet_NoWeather      seoul      168       35      400.94      407.00      -6.05 2.872176e-01      0.2872        False
NeuralProphet vs NeuralProphet_NoWeather     london        6      980     1204.28     1247.18     -42.89 1.282013e-17      0.0000         True
NeuralProphet vs NeuralProphet_NoWeather     london       24      245      644.11      630.45      13.66 5.077067e-03      0.0254         True

## 3. Pooled test (across all cities and horizons)
Single test per comparison pair using all fold-level observations pooled.

In [4]:
for model_a, model_b in COMPARISONS:
    a_all = df_detailed[df_detailed['model'] == model_a].sort_values(
        ['dataset','horizon','fold'])['MAE'].values
    b_all = df_detailed[df_detailed['model'] == model_b].sort_values(
        ['dataset','horizon','fold'])['MAE'].values

    if len(a_all) == 0 or len(b_all) == 0:
        print(f"{model_a} vs {model_b}: one model missing, skipping")
        continue
    if len(a_all) != len(b_all):
        print(f"{model_a} vs {model_b}: unequal lengths ({len(a_all)} vs {len(b_all)}), skipping")
        continue

    stat, p = wilcoxon(a_all, b_all, alternative='two-sided')
    median_diff = np.median(a_all - b_all)
    direction = "A better" if median_diff < 0 else "B better"

    print(f"\n{model_a} vs {model_b}")
    print(f"  n pairs    : {len(a_all)}")
    print(f"  median diff: {median_diff:.2f} ({direction})")
    print(f"  statistic  : {stat:.1f}")
    print(f"  p-value    : {p:.4f}")
    print(f"  significant: {p < 0.05}")


TabPFN vs TabPFN_NoWeather
  n pairs    : 4146
  median diff: -5.62 (A better)
  statistic  : 2907208.0
  p-value    : 0.0000
  significant: True

TabPFN vs XGBoost
  n pairs    : 4146
  median diff: -7.79 (A better)
  statistic  : 3135394.0
  p-value    : 0.0000
  significant: True

NeuralProphet vs NeuralProphet_NoWeather
  n pairs    : 4146
  median diff: -7.63 (A better)
  statistic  : 3423809.0
  p-value    : 0.0000
  significant: True


## 4. Save results

In [5]:
Path('../results/tables/cross_city').mkdir(parents=True, exist_ok=True)
results_corrected.to_csv(
    '../results/tables/cross_city/wilcoxon_results.csv', index=False
)
print("Saved: wilcoxon_results.csv")

Saved: wilcoxon_results.csv
